In [ ]:
from IPython.utils.syspathcontext import appended_to_syspath
import requests
import json
import re
import pandas as pd
import numpy as np

def get_uniprot(accession):

  endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
  http_function = requests.get
  http_args = {'params': {'accession': accession}}
  return http_function(endpoint, **http_args)

def uniprot_parse_response(resp):
    if not resp.ok:
        raise ValueError(f"HTTP error: {resp.status_code}")

    try:
        resp = resp.json()
    except ValueError:
        raise ValueError("Invalid JSON response")

    acc = resp.get('primaryAccession')
    if not acc:
        raise ValueError("Missing primaryAccession in response")

    species = resp.get('organism', {}).get('scientificName')
    gene = resp.get('genes', [])
    seq = resp.get('sequence', {})

    if not seq:
        raise ValueError("Missing sequence information")

    return {
        acc: {
            'organism': species,
            'geneInfo': gene,
            'sequenceInfo': seq,
            'type': 'protein'
        }
    }

def get_ensembl(id):
    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
    http_function = requests.get
    http_args = {
        'headers': {'Content-Type': 'application/json'}
    }

    return http_function(endpoint, **http_args)


def ensembl_parse_response(resp):

    if not resp.ok:
        raise ValueError(f"HTTP error: {resp.status_code}")

    try:
        resp = resp.json()
    except ValueError:
        raise ValueError("Invalid JSON response from Ensembl")

    gene_id = resp.get('id')
    if not gene_id:
        raise ValueError("Missing 'id' field in Ensembl response")

    output = {
        gene_id: {
            'object_type': resp.get('object_type', None),
            'species': resp.get('species', None),
            'assembly_name': resp.get('assembly_name', None),
            'biotype': resp.get('biotype', None),
            'display_name': resp.get('display_name', None),
            'id': gene_id,
            'db_type': resp.get('db_type', None),
            'description': resp.get('description', None),
            'source': resp.get('source', None),
            'canonical_transcript': resp.get('canonical_transcript', None)
        }
    }

    return output


def main(ids: list):
    result = {}

    ensembl_reg = r'^ENS[A-Z]+[0-9]{11}$'
    uniprot_reg = r'^[OPQ][0-9][A-Z0-9]{3}[0-9]$|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$'

    for i in ids:
        try:
            if re.match(ensembl_reg, i):
                result[i] = ensembl_parse_response(get_ensembl(i))[i]
            elif re.match(uniprot_reg, i):
                result[i] = uniprot_parse_response(get_uniprot(i))[i]
            else:
                result[i] = {'error': 'unknown database'}
        except Exception as e:
            result[i] = {'error': str(e)}

    df = pd.DataFrame.from_dict(result, orient='index')
    return df

In [ ]:
get_uniprot('P11473')


<Response [200]>

In [ ]:
get_uniprot('helloworld')

<Response [400]>

In [ ]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [ ]:
uniprot_parse_response(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'}}

In [ ]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [ ]:
get_ensembl('helloworld')

<Response [400]>

In [ ]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [ ]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [ ]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'species': 'mus_musculus',
  'assembly_name': 'GRCm39',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'db_type': 'core',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'source': 'ensembl_havana',
  'canonical_transcript': 'ENSMUST00000044620.11'}}

In [ ]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,organism,geneInfo,sequenceInfo,type,error,object_type,species,assembly_name,biotype,display_name,id,db_type,description,source,canonical_transcript
P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRAF,ENSG00000157764,core,"B-Raf proto-oncogene, serine/threonine kinase ...",ensembl_havana,ENST00000646891.2
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRCA2,ENSG00000139618,core,BRCA2 DNA repair associated [Source:HGNC Symbo...,ensembl_havana,ENST00000380152.8
